# **Environment Setup & Data Exploration**

**GPU Verification**

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

**Data Path Exploration**

In [ ]:
import os

data_path = "/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation"
train_path = os.path.join(data_path, "BraTS2020_TrainingData", "MICCAI_BraTS2020_TrainingData")

patients = os.listdir(train_path)
print("Number of patients:", len(patients))
print(patients[:5])

**Single Patient Files Exploration**

In [ ]:
first_patient = patients[0]
first_patient_path = os.path.join(train_path, first_patient)
print("Patient folder:", first_patient)
print("Files inside:", os.listdir(first_patient_path))

**Opening NIfTI File and Shape Verification**

In [ ]:
import nibabel as nib
import numpy as np

flair_path = os.path.join(first_patient_path, f"{first_patient}_flair.nii")
seg_path = os.path.join(first_patient_path, f"{first_patient}_seg.nii")

flair_img = nib.load(flair_path)
seg_img = nib.load(seg_path)

flair_data = flair_img.get_fdata()
seg_data = seg_img.get_fdata()

print("Flair shape:", flair_data.shape)
print("Seg shape:", seg_data.shape)
print("Unique values in seg:", np.unique(seg_data))

**Visualization of All Modalities**

In [ ]:
import matplotlib.pyplot as plt

t1_path = os.path.join(first_patient_path, f"{first_patient}_t1.nii")
t1ce_path = os.path.join(first_patient_path, f"{first_patient}_t1ce.nii")
t2_path = os.path.join(first_patient_path, f"{first_patient}_t2.nii")

t1_data = nib.load(t1_path).get_fdata()
t1ce_data = nib.load(t1ce_path).get_fdata()
t2_data = nib.load(t2_path).get_fdata()

slice_idx = 75

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
modalities = [flair_data, t1_data, t1ce_data, t2_data, seg_data]
titles = ["FLAIR", "T1", "T1CE", "T2", "Segmentation"]

for ax, mod, title in zip(axes, modalities, titles):
    cmap = "jet" if title == "Segmentation" else "gray"
    ax.imshow(mod[:, :, slice_idx], cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

**Binary Mask Visualization**

In [ ]:
seg_binary = (seg_data > 0).astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(seg_data[:, :, slice_idx], cmap="jet")
axes[0].set_title("Multi-class (original)")
axes[0].axis("off")

axes[1].imshow(seg_binary[:, :, slice_idx], cmap="gray")
axes[1].set_title("Binary (tumor vs background)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

**Clone/Pull Repository**

In [ ]:
!git clone https://github.com/alshafeay/Brain-Tumor-Segmentation.git
%cd Brain-Tumor-Segmentation

In [ ]:
!git pull origin main

# **Data Preprocessing & Pipeline Preparation**

**Dataset Building**

In [ ]:
import sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(('datasets', 'losses', 'train', 'metrics', 'evaluate', 'early_stopping', 'models')):
        del sys.modules[mod_name]

from datasets import BraTSDataset

dataset = BraTSDataset(data_root=train_path, min_tumor_ratio=0.01)
print("Dataset size:", len(dataset))

image, mask = dataset[0]
print("Image shape:", image.shape)
print("Mask shape:", mask.shape)
print("Unique mask values:", torch.unique(mask))

**Train/Val Split (Patient-Level)**

In [ ]:
import random
import copy

random.seed(42)

valid_patients = list(set([p for p, _ in dataset.index]))
random.shuffle(valid_patients)

split_idx = int(0.85 * len(valid_patients))
train_patient_ids = set(valid_patients[:split_idx])
val_patient_ids = set(valid_patients[split_idx:])

train_index = [(p, s) for p, s in dataset.index if p in train_patient_ids]
val_index = [(p, s) for p, s in dataset.index if p in val_patient_ids]

train_dataset = copy.copy(dataset)
train_dataset.index = train_index

val_dataset = copy.copy(dataset)
val_dataset.index = val_index

print("Train patients:", len(train_patient_ids))
print("Val patients:", len(val_patient_ids))
print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))

**DataLoaders**

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 8

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

images, masks = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch masks shape:", masks.shape)

# **Model Architecture & Training Setup**

**Model Architecture Building & Verification**

In [ ]:
from models.swin_unet import SwinUNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SwinUNet(in_channels=4, num_classes=1).to(device)

sample_images, sample_masks = next(iter(train_loader))
sample_images = sample_images.to(device)

output = model(sample_images)
print("Input shape:", sample_images.shape)
print("Output shape:", output.shape)
print("Target mask shape:", sample_masks.shape)

**Importing Loss Functions & Training Functions**

In [ ]:
from losses import CombinedLoss, dice_score
from train import train_one_epoch, validate_one_epoch
from early_stopping import EarlyStopping

**Optimizer Setup & Training Loop with Early Stopping**

In [ ]:
from torch.optim import AdamW
from torch.amp import GradScaler

encoder_params = list(model.encoder.parameters())
decoder_params = (
    list(model.decoder1.parameters())
    + list(model.decoder2.parameters())
    + list(model.decoder3.parameters())
    + list(model.final_expand.parameters())
)

optimizer = AdamW([
    {"params": encoder_params, "lr": 1e-5},
    {"params": decoder_params, "lr": 1e-4},
], weight_decay=1e-4)

criterion = CombinedLoss(dice_weight=0.5, bce_weight=0.5)
scaler = GradScaler('cuda')

early_stopper = EarlyStopping(patience=5, min_delta=0.001)

NUM_EPOCHS = 50
best_dice = 0.0

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
    val_loss, val_dice = validate_one_epoch(model, val_loader, criterion, device)

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Dice: {val_dice:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), "best_model.pt")
        print(f"Saved new best model with Dice: {best_dice:.4f}")

    if early_stopper(val_dice):
        print(f"Early stopping triggered at epoch {epoch+1}. Best Dice: {best_dice:.4f}")
        break

# **Evaluation & Results Visualization**

**Loss Curve Visualization**

In [ ]:
history_train_loss = []
history_val_loss = []
history_val_dice = []

**Full Evaluation with All Metrics**

In [ ]:
for mod_name in list(sys.modules.keys()):
    if mod_name in ('metrics', 'evaluate') or mod_name.startswith(('metrics.', 'evaluate.')):
        del sys.modules[mod_name]

from evaluate import evaluate_model

model.load_state_dict(torch.load("best_model.pt"))

results = evaluate_model(model, val_loader, device)

print("Evaluation Results:")
for metric_name, value in results.items():
    print(f"{metric_name}: {value:.4f}")

**Visual Comparison - Visualizing Results**

In [ ]:
model.eval()

sample_images, sample_masks = next(iter(val_loader))
sample_images = sample_images.to(device)

with torch.no_grad():
    outputs = model(sample_images)
    preds = (torch.sigmoid(outputs) > 0.5).float()

num_samples = 4
fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4 * num_samples))

for i in range(num_samples):
    img_display = sample_images[i, 0].cpu().numpy()
    gt_display = sample_masks[i, 0].cpu().numpy()
    pred_display = preds[i, 0].cpu().numpy()

    axes[i, 0].imshow(img_display, cmap="gray")
    axes[i, 0].set_title("Input (FLAIR channel)")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(gt_display, cmap="gray")
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(pred_display, cmap="gray")
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import FileLink
FileLink("best_model.pt")